In [1]:
import sys
sys.path.insert(0, ".")

from config import MODELS, BFCL_CATEGORIES
from data.ifeval_loader import load_ifeval
from inference.ifeval_runner import run_ifeval_single, run_ifeval_category
from evaluation.ifeval_evaluator import evaluate_ifeval_response, compute_ifeval_metrics, IFEvalMetrics
from tqdm.notebook import tqdm
import pandas as pd
import json
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

pd.set_option("display.max_colwidth", None)
samples = load_ifeval()
print(f"All imports OK. IFEval prompts: {len(samples)}")
print(f"Models: {[m['label'] for m in MODELS]}")


All imports OK. IFEval prompts: 541
Models: ['Granite 4.1 8B', 'Granite 4.0', 'Llama 3.1 8B', 'Qwen 2.5 7B', 'Mistral 7B']


In [ ]:
print("Smoke test: 3 prompts on granite4.1:8b\n")
for s in samples[:3]:
    r = run_ifeval_single("granite4.1:8b", s)
    ev = evaluate_ifeval_response(r["response"], s["instruction_id_list"], s["kwargs"])
    print(f"ID: {s['id']} | pass={ev.prompt_strict} | tokens={r['completion_tokens']} | {r['latency_ms']:.0f}ms")
    print(f"  Instructions: {s['instruction_id_list']}")
    print(f"  Response: {r['response'][:120]}...")
    print()


In [ ]:
all_ifeval_results = {}

for model in MODELS:
    tag, label = model["tag"], model["label"]
    print(f"\n{'='*50}\nRunning: {label} ({tag})\n{'='*50}")
    with tqdm(total=len(samples), desc=label) as pbar:
        results = run_ifeval_category(tag, samples, progress=pbar)
    all_ifeval_results[label] = results
    done = sum(1 for r in results if not r.get("error"))
    print(f"  Completed: {done}/{len(samples)}")

print("\nAll inference complete.")


In [4]:
# Cell 3b — Load from cache (use instead of Cell 3 after kernel restart)
all_ifeval_results = {}
for model in MODELS:
    tag, label = model["tag"], model["label"]
    safe_tag = tag.replace(":", "_").replace("/", "_")
    model_dir = Path("results_ifeval") / safe_tag
    results = [json.loads(f.read_text()) for f in sorted(model_dir.glob("ifeval_*.json"))]
    all_ifeval_results[label] = results
    print(f"{label}: {len(results)} results loaded")
print("all_ifeval_results ready.")


Granite 4.1 8B: 541 results loaded
Granite 4.0: 541 results loaded


Llama 3.1 8B: 541 results loaded


Qwen 2.5 7B: 541 results loaded
Mistral 7B: 541 results loaded
all_ifeval_results ready.


In [5]:
all_ifeval_metrics = {}
for model in MODELS:
    label = model["label"]
    m = compute_ifeval_metrics(all_ifeval_results[label], samples)
    all_ifeval_metrics[label] = m
    print(f"{label}: prompt_acc={m.prompt_strict_acc:.1%}  instr_acc={m.instruction_strict_acc:.1%}  avg_tokens={m.avg_tokens:.0f}")


Granite 4.1 8B: prompt_acc=79.7%  instr_acc=85.1%  avg_tokens=299


Granite 4.0: prompt_acc=77.1%  instr_acc=83.5%  avg_tokens=277
Llama 3.1 8B: prompt_acc=70.8%  instr_acc=78.3%  avg_tokens=299


Qwen 2.5 7B: prompt_acc=70.2%  instr_acc=78.5%  avg_tokens=271
Mistral 7B: prompt_acc=46.8%  instr_acc=56.8%  avg_tokens=345


In [6]:
rows = []
for model in MODELS:
    label = model["label"]
    m = all_ifeval_metrics[label]
    rows.append({
        "Model": label,
        "Prompt Accuracy": f"{m.prompt_strict_acc:.1%}",
        "Instruction Accuracy": f"{m.instruction_strict_acc:.1%}",
        "Avg Tokens": round(m.avg_tokens, 0),
        "Median Latency (ms)": round(m.median_latency_ms, 0),
    })
df = pd.DataFrame(rows).set_index("Model")
print("=== IFEval Phase 2 Results ===")
display(df)

print("\nIBM published Granite 4.1 8B IFEval score = 87.06%")
print("(Instruction-level accuracy)")


=== IFEval Phase 2 Results ===


,Prompt Accuracy,Instruction Accuracy,Avg Tokens,Median Latency (ms)
Model,,,,
Granite 4.1 8B,79.7%,85.1%,299.0,9531.0
Granite 4.0,77.1%,83.5%,277.0,4009.0
Llama 3.1 8B,70.8%,78.3%,299.0,10611.0
Qwen 2.5 7B,70.2%,78.5%,271.0,8022.0
Mistral 7B,46.8%,56.8%,345.0,11399.0



IBM published Granite 4.1 8B IFEval score = 87.06%
(Instruction-level accuracy)


In [7]:
PALETTE = sns.color_palette("colorblind", 5)
models = [m["label"] for m in MODELS]
colors = {label: PALETTE[i] for i, label in enumerate(models)}

prompt_accs = [all_ifeval_metrics[m["label"]].prompt_strict_acc * 100 for m in MODELS]
instr_accs  = [all_ifeval_metrics[m["label"]].instruction_strict_acc * 100 for m in MODELS]

x = np.arange(len(models))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width/2, prompt_accs, width, label="Prompt Accuracy", color=[colors[m] for m in models], alpha=0.9)
ax.bar(x + width/2, instr_accs,  width, label="Instruction Accuracy", color=[colors[m] for m in models], alpha=0.5)
ax.set_ylabel("Accuracy (%)")
ax.set_title("IFEval — Instruction Following Accuracy")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15)
ax.set_ylim(0, 100)
ax.axhline(87.06, color="red", linestyle="--", alpha=0.7, label="IBM published (87.06%)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results_ifeval/ifeval_accuracy.png", dpi=150)
plt.show()
print("Saved: results_ifeval/ifeval_accuracy.png")


Saved: results_ifeval/ifeval_accuracy.png


/var/folders/3w/rtx1bsc114g39997z8xxlt500000gn/T/ipykernel_73456/2500337415.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Compare rankings between tool calling (Phase 1) and instruction following (Phase 2)
phase1_overall = {
    "Granite 4.1 8B": 42.2,
    "Granite 4.0":    34.8,
    "Llama 3.1 8B":   40.4,
    "Qwen 2.5 7B":    42.8,
    "Mistral 7B":     34.9,
}
rows = []
for model in MODELS:
    label = model["label"]
    m = all_ifeval_metrics[label]
    rows.append({
        "Model": label,
        "Phase 1 Tool Calling": f"{phase1_overall.get(label, 0):.1f}%",
        "Phase 2 Instruction Following": f"{m.prompt_strict_acc:.1%}",
    })
df2 = pd.DataFrame(rows).set_index("Model")
print("=== Phase 1 vs Phase 2 Comparison ===")
display(df2)


=== Phase 1 vs Phase 2 Comparison ===


,Phase 1 Tool Calling,Phase 2 Instruction Following
Model,,
Granite 4.1 8B,42.2%,79.7%
Granite 4.0,34.8%,77.1%
Llama 3.1 8B,40.4%,70.8%
Qwen 2.5 7B,42.8%,70.2%
Mistral 7B,34.9%,46.8%
